# Segment query: Mô hình dựa trên mệnh đề + SkinCAP

Notebook này là workflow Colab cho biến thể truy vấn token-level `proposition_segments` của `main_proposition.py` trên SkinCAP-47.

1. Clone nhánh có code segment query và cài dependency.
2. Kiểm tra GPU, Drive, ClinicalBERT, dữ liệu và knowledge base.
3. Tạo hoặc chọn thư mục chạy, rồi ghi cấu hình 30 epoch.
4. Huấn luyện/tiếp tục `main_proposition.py` với truy vấn token-level.
5. Kiểm tra checkpoint và metric đã lưu.
6. So sánh map bằng chứng P1, P2 và Segments.

**Trước khi chạy:** chọn **Runtime → Change runtime type → GPU**.

> Mỗi thí nghiệm mới được đặt trong `Proposition-outputs/skin-proposition-segments/YYYYMMDD_HHMMSS`. Để resume, giữ nguyên cả `EXPERIMENT_NAME` và `RUN_ID`.

## 1. Chuẩn bị Python và repository

Clone nhánh `RAD-colab-segment-query`. Nếu repository đã có trong phiên Colab hiện tại, ô này chuyển sang đúng nhánh và kéo commit mới nhất.

In [ ]:
import subprocess
import sys
from pathlib import Path

print("Python:", sys.version)

REPO_DIR = Path("/content/RAD-colab")
REPO_URL = "https://github.com/trisle0805/RAD-colab.git"
BRANCH = "RAD-colab-segment-query"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-3"], check=True)
assert (REPO_DIR / "main_proposition.py").is_file(), "Sai nhánh: không thấy main_proposition.py"

COLAB_REQUIREMENTS = next(
    (path for path in (
        REPO_DIR / "requirements/colab-skin.txt",
        REPO_DIR / "requirements-colab-skin.txt",
    ) if path.is_file()),
    None,
)
assert COLAB_REQUIREMENTS is not None, "Không tìm thấy file dependency Colab cho SkinCAP."
print(f"✅ Repository: {REPO_DIR} (nhánh {BRANCH})")
print("Requirements:", COLAB_REQUIREMENTS.relative_to(REPO_DIR))

## 2. Cài dependency và kiểm tra GPU

`requirements/colab-skin.txt` không cài lại `torch`/`torchvision`, để giữ cặp CUDA tương thích mà Colab đang cung cấp.

In [ ]:
%cd /content/RAD-colab
%pip install -q -r {COLAB_REQUIREMENTS}

import importlib
from importlib.metadata import version

modules = {
    "yaml": "PyYAML",
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "tqdm": "tqdm",
    "cv2": "opencv-python-headless",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "skimage": "scikit-image",
    "pydicom": "pydicom",
    "nibabel": "nibabel",
    "tensorboardX": "tensorboardX",
    "transformers": "transformers",
    "timm": "timm",
    "einops": "einops",
    "peft": "peft",
    "matplotlib": "matplotlib",
}

for module, package in modules.items():
    importlib.import_module(module)
    print(f"✅ {module:15} {version(package)}")

import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 3. Mount Drive, tải ClinicalBERT khi cần và chọn lần chạy

Chỉ sửa các đường dẫn riêng nếu vị trí dữ liệu hoặc output trên Drive thay đổi. Nếu `/content/models/Bio_ClinicalBERT` chưa tồn tại trong runtime Colab hiện tại, notebook sẽ tải model gốc `emilyalsentzer/Bio_ClinicalBERT` từ Hugging Face vào đó. Runtime Colab là tạm thời nên việc tải lại sau khi reset runtime là bình thường.

- Để `RUN_ID = None` khi chủ động bắt đầu một thí nghiệm mới. Notebook tạo và in một timestamp dạng `YYYYMMDD_HHMMSS`.
- Khi Colab ngắt hoặc reset runtime, dán timestamp đã in vào `RUN_ID` để tiếp tục **đúng** lần chạy đó.
- Không để `RUN_ID = None` khi muốn resume, vì điều này sẽ tạo thí nghiệm mới.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

SKINCAP_ROOT = Path("/content/drive/MyDrive/Cao học/Experiments/SkinCAP")
TRAIN_CSV = SKINCAP_ROOT / "skincap_47_train.csv"
VAL_CSV = SKINCAP_ROOT / "skincap_47_val.csv"
TEST_CSV = SKINCAP_ROOT / "skincap_47_test.csv"
IMAGE_ROOT = SKINCAP_ROOT / "skincap"

BERT_MODEL_ID = "emilyalsentzer/Bio_ClinicalBERT"
BERT_DIR = Path("/content/models/Bio_ClinicalBERT")
if not (BERT_DIR / "config.json").is_file():
    from huggingface_hub import snapshot_download

    BERT_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"Đang tải {BERT_MODEL_ID} vào {BERT_DIR}...")
    snapshot_download(repo_id=BERT_MODEL_ID, local_dir=str(BERT_DIR))
else:
    print(f"✅ Đã có ClinicalBERT: {BERT_DIR}")

# None: tạo một run Segments mới. Muốn resume hoặc chỉ xem kết quả cũ: dán RUN_ID vào đây.
EXPERIMENT_NAME = "skin-proposition-segments"
RUNS_ROOT = Path("/content/drive/MyDrive/Cao học/Proposition-outputs") / EXPERIMENT_NAME
RUN_ID = None

for label, path, check in [
    ("Train CSV", TRAIN_CSV, Path.is_file),
    ("Validation CSV", VAL_CSV, Path.is_file),
    ("Test CSV", TEST_CSV, Path.is_file),
    ("Thư mục ảnh", IMAGE_ROOT, Path.is_dir),
    ("ClinicalBERT", BERT_DIR, Path.is_dir),
]:
    assert check(path), f"Không tìm thấy {label}: {path}"
    print(f"✅ {label}: {path}")

png_count = len(list(IMAGE_ROOT.glob("*.png")))
print("Số ảnh PNG:", png_count)
assert png_count > 0, "Không có ảnh PNG trong IMAGE_ROOT."

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    RUN_DIR = RUNS_ROOT / RUN_ID
    RUN_DIR.mkdir(parents=False, exist_ok=False)
    print(f"✅ Tạo lần chạy Segments mới: {RUN_ID} (UTC)")
    print("⚠️ Lưu EXPERIMENT_NAME và RUN_ID này để resume sau khi runtime reset.")
else:
    RUN_DIR = RUNS_ROOT / RUN_ID
    assert RUN_DIR.is_dir(), f"Không tìm thấy run trên Drive: {RUN_DIR}"
    latest_checkpoint = RUN_DIR / "checkpoints" / "latest.pt"
    best_checkpoint = RUN_DIR / "checkpoints" / "best.pt"
    final_metrics = RUN_DIR / "final_metrics.json"
    print(f"✅ Dùng run Segments có sẵn: {RUN_ID}")
    print("Latest checkpoint:", latest_checkpoint if latest_checkpoint.is_file() else "chưa có")
    print("Best checkpoint:", best_checkpoint if best_checkpoint.is_file() else "chưa có")
    print("Final test metrics:", final_metrics if final_metrics.is_file() else "chưa có")

for directory in ("checkpoints", "metrics", "predictions", "checklist"):
    (RUN_DIR / directory).mkdir(exist_ok=True)

print("Run directory:", RUN_DIR)

## 4. Kiểm tra ClinicalBERT và một batch SkinCAP

Ô này không train. Nó xác nhận model BERT cục bộ có hidden size 768 và loader đọc đúng CSV, ảnh và 47 nhãn trước khi bắt đầu thí nghiệm dài.

In [ ]:
%cd /content/RAD-colab

from transformers import AutoConfig, AutoTokenizer
from torch.utils.data import DataLoader
from dataset.dataset_entity import Skin_Train_Dataset
from dataset.test_dataset import Skin_Test_Dataset

bert_config = AutoConfig.from_pretrained(BERT_DIR, local_files_only=True)
AutoTokenizer.from_pretrained(BERT_DIR, local_files_only=True)
assert bert_config.hidden_size == 768, (
    f"ClinicalBERT phải có hidden_size=768, nhận được {bert_config.hidden_size}."
)
print("✅ ClinicalBERT:", bert_config.model_type, "hidden_size=", bert_config.hidden_size)

train_dataset = Skin_Train_Dataset(TRAIN_CSV, image_res=512, image_root=IMAGE_ROOT)
val_dataset = Skin_Test_Dataset(VAL_CSV, image_res=512, image_root=IMAGE_ROOT)
test_dataset = Skin_Test_Dataset(TEST_CSV, image_res=512, image_root=IMAGE_ROOT)
batch = next(iter(DataLoader(train_dataset, batch_size=2, num_workers=0)))

print("Số mẫu train:", len(train_dataset))
print("Số mẫu val :", len(val_dataset))
print("Số mẫu test :", len(test_dataset))
print("Batch image :", tuple(batch["image"].shape))
print("Batch label :", tuple(batch["label"].shape))
assert tuple(batch["image"].shape) == (2, 3, 512, 512)
assert tuple(batch["label"].shape) == (2, 47)
print("✅ Dữ liệu SkinCAP sẵn sàng.")

from proposition.data.knowledge_base import load_proposition_kb
import pandas as pd

KB_PATH = REPO_DIR / "kb_builder/outputs/final/kb_propositions.json"
labels = list(pd.read_csv(TRAIN_CSV, nrows=0).columns[2:])
kb = load_proposition_kb(KB_PATH, labels)
print("Số bệnh:", kb.num_diseases, "| Số mệnh đề:", kb.num_propositions,
      "| Polarity −1:", int((kb.prop_polarity == -1).sum()))
assert kb.num_diseases == 47 and kb.num_propositions == 1436
print("✅ Tập mệnh đề khớp đúng thứ tự 47 nhãn trong CSV.")


## 5. Tạo cấu hình huấn luyện Segments

Cấu hình Segments dùng `batch_size=4`, `grad_accumulation_steps=4`, **30 epoch**, `warmup_epochs=10`. File cấu hình được lưu trong `RUN_DIR`; khi resume, notebook dùng lại file đó để tránh làm thay đổi một lần chạy đang tiếp tục.

In [ ]:
import yaml

FULL_CONFIG = RUN_DIR / "proposition-skin-segments-30ep.yaml"
if FULL_CONFIG.is_file():
    config = yaml.safe_load(FULL_CONFIG.read_text(encoding="utf-8"))
    print(f"✅ Dùng lại cấu hình đã lưu của run: {FULL_CONFIG}")
else:
    with (REPO_DIR / "configs/skin.yaml").open(encoding="utf-8") as file:
        config = yaml.safe_load(file)

    config.update({
        "batch_size": 4,
        "test_batch_size": 4,
        "num_workers": 2,
        "test_num_workers": 2,
        "grad_accumulation_steps": 4,
    })
    config["schedular"].update({
        "epochs": 30,
        "warmup_epochs": 10,
    })
    FULL_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    print(f"✅ Đã tạo cấu hình: {FULL_CONFIG}")

assert config["image_res"] == 512
assert config["batch_size"] == 4 and config["test_batch_size"] == 4
assert config["grad_accumulation_steps"] == 4
assert config["schedular"]["epochs"] == 30 and config["schedular"]["warmup_epochs"] == 10
print(FULL_CONFIG.read_text(encoding="utf-8"))

## 6. Khởi chạy hoặc resume Segments an toàn

Sau mỗi **epoch hoàn chỉnh** (train + validation), code lưu trên Drive checkpoint, metrics và validation predictions. Khi train xong, code test `best.pt` một lần và ghi `final_metrics.json`, `predictions/test_best.npz`, `run_config.json` và thư mục `checklist/`.

Lệnh Segments dùng `--query_level proposition_segments`; ClinicalBERT token query vẫn trainable. Output của tiến trình train được in trực tiếp trong notebook; checkpoint và log trên Drive vẫn được giữ nếu Colab ngắt giữa chừng.

In [ ]:
import os
import subprocess
import sys

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

KB_PATH = REPO_DIR / "kb_builder/outputs/final/kb_propositions.json"
assert KB_PATH.is_file(), f"Không tìm thấy KB: {KB_PATH}"

command = [
    sys.executable, "main_proposition.py",
    "--config", str(FULL_CONFIG),
    "--train_csv", str(TRAIN_CSV),
    "--val_csv", str(VAL_CSV),
    "--test_csv", str(TEST_CSV),
    "--image_root", str(IMAGE_ROOT),
    "--output_dir", str(RUN_DIR),
    "--bert_model_name", str(BERT_DIR),
    "--kb_path", str(KB_PATH),
    "--max_length", "512",
    "--embed_dim", "768",
    "--topk", "1", "2", "3",
    "--seed", "42",
    "--run_id", "SEG_s42",
    "--query_level", "proposition_segments",
    "--query_chunk_size", "2048",
]

print("Sẽ chạy (hoặc resume) Segments:")
print(" ".join(repr(part) if " " in part else part for part in command))
subprocess.run(command, cwd=REPO_DIR, check=True)

## 7. Kiểm tra trạng thái đã lưu trên Drive

Sau khi một epoch hoàn thành, chạy ô này để xem checkpoint có thể resume, metrics mới nhất và phần cuối của log. Ô này chỉ đọc file; không bắt đầu huấn luyện.

In [ ]:
import json
import pandas as pd

latest_checkpoint = RUN_DIR / "checkpoints" / "latest.pt"
history_path = RUN_DIR / "metrics" / "metrics_history.csv"
latest_metrics_path = RUN_DIR / "metrics" / "latest_metrics.json"
log_path = RUN_DIR / "logs" / "training.log"
best_checkpoint = RUN_DIR / "checkpoints" / "best.pt"
final_metrics_path = RUN_DIR / "final_metrics.json"

print("Run directory:", RUN_DIR)
print("Latest checkpoint:", latest_checkpoint if latest_checkpoint.is_file() else "chưa có")
print("Best checkpoint:", best_checkpoint if best_checkpoint.is_file() else "chưa có")
print("Milestone checkpoints:", [path.name for path in sorted((RUN_DIR / "checkpoints").glob("epoch_*.pt"))])

if history_path.is_file():
    history = pd.read_csv(history_path)
    display(history.tail(5))
else:
    print("ℹ️ Chưa có metrics_history.csv; epoch đầu tiên chưa hoàn thành.")

if latest_metrics_path.is_file():
    print("\nLatest metrics:")
    print(json.dumps(json.loads(latest_metrics_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))

if log_path.is_file():
    print("\n20 dòng log cuối:")
    print("\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]))
if final_metrics_path.is_file():
    print("\nFinal test metrics:")
    print(json.dumps(json.loads(final_metrics_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
else:
    print("\nChưa có final_metrics.json; file này chỉ được tạo sau khi train xong và test best.pt một lần.")


## 8. So sánh với RAD gốc

Đọc `final_metrics.json` của lần chạy RAD (`20261005_091021`) và của lần chạy này. RAD báo ba đầu ra: `fused` (đầu ra chính thức của RAD), `guideline` và `label`. Mô hình mệnh đề báo `proposition` (đầu ra chính) và `label_aux` (chỉ tham khảo).


In [ ]:
import json
from pathlib import Path

import pandas as pd

RAD_RUN_DIR = Path(
    "/content/drive/MyDrive/Cao học/RAD-outputs/skin-rad-20ep/20261007_111918"
)
RAD_METRICS_PATH = RAD_RUN_DIR / "final_metrics.json"
PROPOSITION_METRICS_PATH = RUN_DIR / "final_metrics.json"

assert PROPOSITION_METRICS_PATH.is_file(), (
    "Chưa có final_metrics.json của run mệnh đề. Hãy hoàn tất ô huấn luyện trước."
)
assert RAD_METRICS_PATH.is_file(), f"Chưa có final_metrics RAD: {RAD_METRICS_PATH}."

rad = json.loads(RAD_METRICS_PATH.read_text(encoding="utf-8"))
prop = json.loads(PROPOSITION_METRICS_PATH.read_text(encoding="utf-8"))

METRICS = ["top1_accuracy", "top2_accuracy", "top3_accuracy", "mrr_at_2", "mrr_at_3",
           "macro_mrr_at_3", "macro_f1", "balanced_accuracy", "macro_auc", "mAP"]
LEGACY = ["mean_f1", "mean_precision", "mean_recall", "mean_accuracy", "subset_accuracy"]

def row(block):
    values = {name: block[name] for name in METRICS}
    values.update({f"rad_{name}": block["rad_legacy_metrics"][name] for name in LEGACY})
    return values

columns = {
    "RAD fused": row(rad["fused"]),
    "RAD guideline": row(rad["guideline"]),
    "Mệnh đề (chính)": row(prop["proposition"]),
}
if "label_aux" in prop:
    columns["Mệnh đề label_aux"] = row(prop["label_aux"])
table = pd.DataFrame(columns).round(4)
table["Δ (mệnh đề − RAD fused)"] = (table["Mệnh đề (chính)"] - table["RAD fused"]).round(4)
display(table)
print("Best epoch — RAD:", rad.get("best_epoch"), "| Mệnh đề:", prop.get("best_epoch"))

## 8. Xem một ca bệnh từ checkpoint tốt nhất

Phần này chỉ đọc các artefact test đã được tạo sau khi `main_proposition.py` nạp `checkpoints/best.pt`. Không huấn luyện lại và không nạp checkpoint lần nữa.

Mặc định, notebook chọn mẫu đầu tiên trong danh sách các mẫu có **full attention** đã lưu từ lượt test của `best.pt`. Có thể đổi `SAMPLE_INDEX` thành một chỉ số khác trong `AVAILABLE_SAMPLES` để xem một bệnh nhân khác.

> Trong mode Segments, file evidence là `test_segment_evidence.npz`; các trục attention vẫn là sample, proposition và memory token.

In [ ]:
from IPython.display import IFrame, display
import json
import subprocess
import sys
import numpy as np

CHECKLIST_DIR = RUN_DIR / "checklist"
EVIDENCE_PATH = CHECKLIST_DIR / "test_segment_evidence.npz"
SCHEMA_PATH = CHECKLIST_DIR / "test_segment_evidence.schema.json"
BEST_CHECKPOINT = RUN_DIR / "checkpoints" / "best.pt"

assert BEST_CHECKPOINT.is_file(), f"Không tìm thấy checkpoint tốt nhất: {BEST_CHECKPOINT}"
assert EVIDENCE_PATH.is_file(), (
    "Không tìm thấy evidence test. Hãy chạy hoàn chỉnh ô huấn luyện để chương trình "
    "test checkpoint tốt nhất và tạo thư mục checklist/."
)
assert SCHEMA_PATH.is_file(), f"Không tìm thấy schema: {SCHEMA_PATH}"

with np.load(EVIDENCE_PATH, allow_pickle=False) as evidence:
    AVAILABLE_SAMPLES = evidence["full_attention_sample_indices"].astype(int).tolist()

assert AVAILABLE_SAMPLES, (
    "Artefact không chứa full attention. Khi chạy train cần giữ --full_attention_samples lớn hơn 0."
)
print("Checkpoint dùng để tạo evidence:", BEST_CHECKPOINT)
print("Các mẫu có full attention:", AVAILABLE_SAMPLES)

# None: tự chọn mẫu đầu tiên có full attention từ lượt test dùng best.pt.
# Muốn xem ca khác: thay bằng một giá trị trong AVAILABLE_SAMPLES, ví dụ SAMPLE_INDEX = AVAILABLE_SAMPLES[1].
SAMPLE_INDEX = None
if SAMPLE_INDEX is None:
    SAMPLE_INDEX = AVAILABLE_SAMPLES[0]

assert SAMPLE_INDEX in AVAILABLE_SAMPLES, (
    f"Mẫu {SAMPLE_INDEX} không có full attention. Chọn một trong: {AVAILABLE_SAMPLES}"
)
print("Mẫu đang xem:", SAMPLE_INDEX)

## 9. So sánh map bằng chứng P1, P2 và Segments

Ô này chạy `segment_map_analysis.py` trên hai evidence cố định của P1/P2 và evidence của `RUN_DIR`, sau đó in một bảng ba cột.

In [ ]:
import json

import pandas as pd

P1_EVIDENCE = Path(
    "/content/drive/MyDrive/Cao học/Proposition-outputs/skin-proposition-20ep/"
    "20261007_112004/checklist/test_proposition_evidence.npz"
)
P2_EVIDENCE = Path(
    "/content/drive/MyDrive/Cao học/Proposition-outputs/skin-proposition-P2-frozenq/"
    "20261008_121605/checklist/test_proposition_evidence.npz"
)
SEGMENT_EVIDENCE = RUN_DIR / "checklist" / "test_segment_evidence.npz"

for evidence_path in (P1_EVIDENCE, P2_EVIDENCE, SEGMENT_EVIDENCE):
    assert evidence_path.is_file(), f"Chưa có evidence: {evidence_path}"
    subprocess.run(
        [
            sys.executable,
            "proposition/visualization/segment_map_analysis.py",
            "--evidence", str(evidence_path),
        ],
        cwd=REPO_DIR,
        check=True,
    )

fields = [
    "within_case_same_disease_corr",
    "same_prop_across_cases_corr",
    "image_max_over_mean",
    "image_mass_share",
    "num_cases",
    "num_props_used",
]
analysis_paths = [
    P1_EVIDENCE.parent / "segment_map_analysis.json",
    P2_EVIDENCE.parent / "segment_map_analysis.json",
    SEGMENT_EVIDENCE.parent / "segment_map_analysis.json",
]
analyses = [json.loads(path.read_text(encoding="utf-8")) for path in analysis_paths]
comparison = pd.DataFrame({
    "P1": {field: analyses[0][field] for field in fields},
    "P2": {field: analyses[1][field] for field in fields},
    "Segments": {field: analyses[2][field] for field in fields},
})
display(comparison.round(4))